# 🗂️ disk_denuncia.csv — Structural Transformation

The raw CSV is a **flattened representation of nested data**: each `denuncia` spans
multiple rows to accommodate its related `orgaos`, `assuntos`, and `envolvidos`.

This notebook normalizes it into **4 clean, relational DataFrames**:

| DataFrame | Description | Key |
|---|---|---|
| `df_denuncias` | One row per complaint (18 003 rows) | `numero_denuncia` |
| `df_orgaos` | Agencies notified per complaint | `numero_denuncia` |
| `df_assuntos` | Crime subjects/types per complaint | `numero_denuncia` |
| `df_envolvidos` | Suspects/people involved | `numero_denuncia` |

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

FILE = 'disk_denuncia.csv'   # ← adjust path if needed

---
## 1. Load & Repair Raw File

In [2]:
raw = pd.read_csv(FILE, encoding='latin-1', sep=';', low_memory=False)
print(f'Raw shape: {raw.shape}')   # (83 549, 48)

# ── Forward-fill numero_denuncia so every sub-row knows its parent ──
raw['numero_denuncia'] = raw['numero_denuncia'].ffill()

# ── Fix lat/lon: Brazil uses comma as decimal separator ──
for col in ['latitude', 'longitude']:
    raw[col] = (
        raw[col]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .replace('nan', np.nan)
    )
    raw[col] = pd.to_numeric(raw[col], errors='coerce')

print('Done.')

Raw shape: (83549, 48)
Done.


---
## 2. `df_denuncias` — One row per complaint

In [3]:
# These columns are stable across all sub-rows of a denuncia
DENUNCIA_COLS = [
    'numero_denuncia',
    'id_denuncia',
    'data_denuncia',
    'data_difusao',
    'timestamp_insercao',
    'status_denuncia',
    'tipo_logradouro',
    'logradouro',
    'numero_logradouro',
    'complemento_logradouro',
    'bairro_logradouro',
    'subbairro_logradouro',
    'cep_logradouro',
    'referencia_logradouro',
    'municipio',
    'estado',
    'latitude',
    'longitude',
    # Primary classification (filled on anchor row)
    'id_classe',
    'classe',
    'id_tipo',
    'tipo',
    'assunto_principal',
    'relato_redacted',
]

# Take the first (anchor) row of each denuncia
df_denuncias = (
    raw[raw['id_denuncia'].notna()]   # anchor rows have id_denuncia filled
    [DENUNCIA_COLS]
    .drop_duplicates(subset='numero_denuncia')
    .reset_index(drop=True)
)

# ── Type casting ──
df_denuncias['id_denuncia']   = df_denuncias['id_denuncia'].astype('Int64')
df_denuncias['id_classe']     = df_denuncias['id_classe'].astype('Int64')
df_denuncias['id_tipo']       = df_denuncias['id_tipo'].astype('Int64')
df_denuncias['assunto_principal'] = df_denuncias['assunto_principal'].astype('Int64')
df_denuncias['cep_logradouro']    = df_denuncias['cep_logradouro'].astype('Int64')

for col in ['data_denuncia', 'data_difusao', 'timestamp_insercao']:
    df_denuncias[col] = pd.to_datetime(df_denuncias[col], errors='coerce')

# ── Strip extra whitespace from string columns ──
str_cols = df_denuncias.select_dtypes('object').columns
df_denuncias[str_cols] = df_denuncias[str_cols].apply(lambda s: s.str.strip())

print(f'df_denuncias shape: {df_denuncias.shape}')
df_denuncias.info()
df_denuncias.head(3)

df_denuncias shape: (18003, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18003 entries, 0 to 18002
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   numero_denuncia         18003 non-null  object        
 1   id_denuncia             18003 non-null  Int64         
 2   data_denuncia           18003 non-null  datetime64[ns]
 3   data_difusao            18003 non-null  datetime64[ns]
 4   timestamp_insercao      18003 non-null  datetime64[ns]
 5   status_denuncia         5118 non-null   object        
 6   tipo_logradouro         17991 non-null  object        
 7   logradouro              18003 non-null  object        
 8   numero_logradouro       7038 non-null   object        
 9   complemento_logradouro  2470 non-null   object        
 10  bairro_logradouro       18003 non-null  object        
 11  subbairro_logradouro    3817 non-null   object        
 12  cep_logradouro

,numero_denuncia,id_denuncia,data_denuncia,data_difusao,timestamp_insercao,status_denuncia,tipo_logradouro,logradouro,numero_logradouro,complemento_logradouro,...,municipio,estado,latitude,longitude,id_classe,classe,id_tipo,tipo,assunto_principal,relato_redacted
0,1024.6.2020,2301680,2020-06-04 08:16:00,2020-06-15 14:57:00,2024-07-15,NaN,R,SANTO CRISTO,NaN,NaN,...,RIO DE JANEIRO,RJ,-22.899555,-43.201388,12,SUBSTÂNCIAS ENTORPECENTES,84,CONSUMO DE DROGAS,1,NA RUA CITADA ESQUINA COM A VIA BINARIO DO [NO...
1,1029.12.2022,2511207,2022-12-07 17:19:00,2022-12-07 22:26:00,2024-07-15,NaN,EST,SETE RIACHOS,240,NaN,...,RIO DE JANEIRO,RJ,-22.867285,-43.509775,12,SUBSTÂNCIAS ENTORPECENTES,84,CONSUMO DE DROGAS,0,NO ENDERECO CITADO PROXIMO A AVENIDA [NOME] SI...
2,1047.10.2019,2240894,2019-10-04 09:33:00,2019-12-26 15:31:00,2024-07-15,NaN,R,BARROS BARRETO,NaN,NaN,...,RIO DE JANEIRO,RJ,-22.859195,-43.254603,12,SUBSTÂNCIAS ENTORPECENTES,84,CONSUMO DE DROGAS,1,NA RUA CITADA NA ALTURA DO N 107 CENTO E [NOME...


## leitura das regiões

In [5]:
pip install geopandas

  Using cached geopandas-1.1.3-py3-none-any.whl.metadata (2.3 kB)
Using cached geopandas-1.1.3-py3-none-any.whl (342 kB)
   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
    --------------------------------------- 0.5/22.9 MB 2.8 MB/s eta 0:00:09
   - -------------------------------------- 1.0/22.9 MB 4.2 MB/s eta 0:00:06
   ---- ----------------------------------- 2.4/22.9 MB 4.1 MB/s eta 0:00:06
   ------- -------------------------------- 4.2/22.9 MB 5.3 MB/s eta 0:00:04
   ------------ --------------------------- 7.1/22.9 MB 7.3 MB/s eta 0:00:03
   -------------- ------------------------- 8.4/22.9 MB 7.3 MB/s eta 0:00:02
   --------------- ------------------------ 8.7/22.9 MB 7.4 MB/s eta 0:00:02
   ------------------- -------------------- 11.0/22.9 MB 6.9 MB/s eta 0:00:02
   ------------------------ --------------- 13.9/22.9 MB 7.7 MB/s eta 0:00:02
   -------------------------- ------------- 15.5/22.9 MB 7.7 MB/s eta 0:00:01
   --------------------------- -----

In [6]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# ── 1. Load the shapefile ──────────────────────────────────────────────────────
gdf_areas = gpd.read_file("../sh_area_forca/areas_forca_municipal.shp")

print("CRS:", gdf_areas.crs)
print("Shape:", gdf_areas.shape)
print("Columns:", gdf_areas.columns.tolist())
display(gdf_areas.head(3))

CRS: EPSG:4326
Shape: (8, 3)
Columns: ['fid', 'nome_subar', 'geometry']


,fid,nome_subar,geometry
0,2.0,Rodoviária - Terminal Gentileza - Estação Leop...,"POLYGON ((-43.21251 -22.89815, -43.21289 -22.8..."
1,9.0,Metrô Botafogo - Rua São Clemente - Rua Volunt...,"POLYGON ((-43.19636 -22.9525, -43.19578 -22.95..."
2,10.0,Jardim de Alah,"POLYGON ((-43.21838 -22.98006, -43.21846 -22.9..."


In [7]:
gdf_areas

,fid,nome_subar,geometry
0,2.0,Rodoviária - Terminal Gentileza - Estação Leop...,"POLYGON ((-43.21251 -22.89815, -43.21289 -22.8..."
1,9.0,Metrô Botafogo - Rua São Clemente - Rua Volunt...,"POLYGON ((-43.19636 -22.9525, -43.19578 -22.95..."
2,10.0,Jardim de Alah,"POLYGON ((-43.21838 -22.98006, -43.21846 -22.9..."
3,11.0,Campo Grande: Estação de Trem - Calçadão,"POLYGON ((-43.56256 -22.90247, -43.56289 -22.9..."
4,12.0,Rio Sul,"POLYGON ((-43.18037 -22.95593, -43.18092 -22.9..."
5,14.0,Praia de Botafogo - Rua Marquês de Abrantes,"POLYGON ((-43.18282 -22.94779, -43.18432 -22.9..."
6,19.0,Estações São Francisco Xavier - Afonso Pena,"POLYGON ((-43.22726 -22.92115, -43.22722 -22.9..."
7,20.0,Presidente Vargas - Campo de Santana - Central...,"POLYGON ((-43.19672 -22.90662, -43.19642 -22.9..."


In [8]:
# ── 2. Reproject to WGS84 (EPSG:4326) if needed ───────────────────────────────
if gdf_areas.crs and gdf_areas.crs.to_epsg() != 4326:
    gdf_areas = gdf_areas.to_crs(epsg=4326)
    print("Reprojected to EPSG:4326")

In [9]:
gdf_areas

,fid,nome_subar,geometry
0,2.0,Rodoviária - Terminal Gentileza - Estação Leop...,"POLYGON ((-43.21251 -22.89815, -43.21289 -22.8..."
1,9.0,Metrô Botafogo - Rua São Clemente - Rua Volunt...,"POLYGON ((-43.19636 -22.9525, -43.19578 -22.95..."
2,10.0,Jardim de Alah,"POLYGON ((-43.21838 -22.98006, -43.21846 -22.9..."
3,11.0,Campo Grande: Estação de Trem - Calçadão,"POLYGON ((-43.56256 -22.90247, -43.56289 -22.9..."
4,12.0,Rio Sul,"POLYGON ((-43.18037 -22.95593, -43.18092 -22.9..."
5,14.0,Praia de Botafogo - Rua Marquês de Abrantes,"POLYGON ((-43.18282 -22.94779, -43.18432 -22.9..."
6,19.0,Estações São Francisco Xavier - Afonso Pena,"POLYGON ((-43.22726 -22.92115, -43.22722 -22.9..."
7,20.0,Presidente Vargas - Campo de Santana - Central...,"POLYGON ((-43.19672 -22.90662, -43.19642 -22.9..."


In [10]:
# ── 3. Build GeoDataFrame from df_denuncias lat/lon ───────────────────────────
df_with_coords = df_denuncias[df_denuncias["latitude"].notna() & df_denuncias["longitude"].notna()].copy()

gdf_denuncias = gpd.GeoDataFrame(
    df_with_coords,
    geometry=gpd.points_from_xy(df_with_coords["longitude"], df_with_coords["latitude"]),
    crs="EPSG:4326"
)

print(f"Denuncias com coordenadas: {len(gdf_denuncias):,} / {len(df_denuncias):,}")

Denuncias com coordenadas: 17,850 / 18,003


In [11]:
# ── 4. Spatial join: assign each point to an area ─────────────────────────────
# Inspect which column holds the area name — adjust AREA_COL if needed
AREA_COL = gdf_areas.columns[0]          # placeholder; will be overridden below
name_candidates = [c for c in gdf_areas.columns
                   if any(kw in c.lower() for kw in ["nome", "name", "area", "regiao", "bairro", "zona"])]
AREA_COL = name_candidates[0] if name_candidates else gdf_areas.columns[0]
print(f"Using area name column: '{AREA_COL}'")

gdf_joined = gpd.sjoin(
    gdf_denuncias,
    gdf_areas[["geometry", AREA_COL]].rename(columns={AREA_COL: "area_forca_municipal"}),
    how="left",
    predicate="within"
)

# Drop the sjoin index column
gdf_joined = gdf_joined.drop(columns=["index_right"], errors="ignore")

# Points that fall outside all polygons get NaN → label explicitly
gdf_joined["area_forca_municipal"] = gdf_joined["area_forca_municipal"].fillna("Fora da área de operação")

print(f"\nDistribuição por área:")
display(gdf_joined["area_forca_municipal"].value_counts().to_frame("denuncias"))

Using area name column: 'nome_subar'

Distribuição por área:


,denuncias
area_forca_municipal,
Fora da área de operação,17078
Presidente Vargas - Campo de Santana - Central do Brasil - Cinelândia,231
Estações São Francisco Xavier - Afonso Pena,146
Rodoviária - Terminal Gentileza - Estação Leopoldina,134
Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria,86
Praia de Botafogo - Rua Marquês de Abrantes,62
Rio Sul,58
Campo Grande: Estação de Trem - Calçadão,38
Jardim de Alah,17


In [12]:
# ── 5. Merge back into df_denuncias (including rows without coordinates) ───────
df_denuncias = df_denuncias.merge(
    gdf_joined[["numero_denuncia", "area_forca_municipal"]],
    on="numero_denuncia",
    how="left"
)

df_denuncias["area_forca_municipal"] = df_denuncias["area_forca_municipal"].fillna("Sem coordenadas")

print("df_denuncias shape after join:", df_denuncias.shape)
display(df_denuncias[["numero_denuncia", "bairro_logradouro", "latitude", "longitude", "area_forca_municipal"]].head(10))

df_denuncias shape after join: (18003, 25)


,numero_denuncia,bairro_logradouro,latitude,longitude,area_forca_municipal
0,1024.6.2020,SANTO CRISTO,-22.899555,-43.201388,Fora da área de operação
1,1029.12.2022,SANTISSIMO,-22.867285,-43.509775,Fora da área de operação
2,1047.10.2019,BONSUCESSO,-22.859195,-43.254603,Fora da área de operação
3,1083.11.2022,TIJUCA,-22.921894,-43.223293,Estações São Francisco Xavier - Afonso Pena
4,115.6.2020,GLORIA,-22.921333,-43.181368,Fora da área de operação
5,1173.1.2025,CASCADURA,-22.884413,-43.331944,Fora da área de operação
6,1243.12.2019,PARADA DE LUCAS,-22.816170,-43.301324,Fora da área de operação
7,1245.9.2020,CAMPO GRANDE,-22.898399,-43.570476,Fora da área de operação
8,1321.2.2026,CAMPO GRANDE,-22.897261,-43.573645,Fora da área de operação
9,1371.3.2026,ROCHA MIRANDA,-22.852650,-43.353520,Fora da área de operação


In [13]:
# ── 6. Summary ─────────────────────────────────────────────────────────────────
print("=== Denúncias por área da Força Municipal ===\n")
summary = (
    df_denuncias
    .groupby("area_forca_municipal")
    .agg(
        total_denuncias=("numero_denuncia", "count"),
        sem_coordenadas=("latitude", lambda x: x.isna().sum()),
    )
    .sort_values("total_denuncias", ascending=False)
)
display(summary)

=== Denúncias por área da Força Municipal ===



,total_denuncias,sem_coordenadas
area_forca_municipal,,
Fora da área de operação,17078,0
Presidente Vargas - Campo de Santana - Central do Brasil - Cinelândia,231,0
Sem coordenadas,153,153
Estações São Francisco Xavier - Afonso Pena,146,0
Rodoviária - Terminal Gentileza - Estação Leopoldina,134,0
Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria,86,0
Praia de Botafogo - Rua Marquês de Abrantes,62,0
Rio Sul,58,0
Campo Grande: Estação de Trem - Calçadão,38,0


In [15]:
import folium
import geopandas as gpd
import json
import random

# ── 1. Garante WGS84 ──────────────────────────────────────────────────────────
gdf = gdf_areas.to_crs(epsg=4326) if gdf_areas.crs.to_epsg() != 4326 else gdf_areas.copy()

# ── 2. Detecta coluna de nome da área ────────────────────────────────────────
name_candidates = [c for c in gdf.columns
                   if any(k in c.lower() for k in ["nome", "name", "area", "regiao", "zona", "setor"])]
AREA_COL = name_candidates[0] if name_candidates else gdf.columns[0]
print(f"Coluna de nome: '{AREA_COL}'")
print(f"Áreas encontradas: {gdf[AREA_COL].tolist()}")

# ── 3. Paleta de cores por área ───────────────────────────────────────────────
PALETTE = [
    "#1D9E75", "#378ADD", "#D85A30", "#7F77DD",
    "#D4537E", "#BA7517", "#639922", "#888780",
    "#5DCAA5", "#85B7EB", "#F0997B", "#AFA9EC",
]
areas = gdf[AREA_COL].tolist()
color_map = {area: PALETTE[i % len(PALETTE)] for i, area in enumerate(areas)}

# ── 4. Centro do mapa (centroide de todas as áreas) ───────────────────────────
center = gdf.dissolve().centroid.iloc[0]
m = folium.Map(
    location=[center.y, center.x],
    zoom_start=12,
    tiles="CartoDB positron",   # mapa limpo em PT
)

# ── 5. Adiciona cada polígono ─────────────────────────────────────────────────
for _, row in gdf.iterrows():
    area_name = row[AREA_COL]
    color     = color_map.get(area_name, "#888780")

    folium.GeoJson(
        data=row["geometry"].__geo_interface__,
        style_function=lambda feat, c=color: {
            "fillColor"   : c,
            "color"       : "#ffffff",
            "weight"      : 1.5,
            "fillOpacity" : 0.45,
        },
        highlight_function=lambda feat, c=color: {
            "fillColor"   : c,
            "color"       : "#ffffff",
            "weight"      : 2.5,
            "fillOpacity" : 0.75,
        },
        tooltip=folium.Tooltip(
            text=f"<b>{area_name}</b>",
            sticky=True,
        ),
    ).add_to(m)

    # Label no centroide do polígono
    cx, cy = row["geometry"].centroid.x, row["geometry"].centroid.y
    folium.Marker(
        location=[cy, cx],
        icon=folium.DivIcon(
            html=f"""<div style="
                font-size:11px; font-weight:600;
                color:#1a1a1a; background:rgba(255,255,255,0.82);
                padding:2px 6px; border-radius:4px;
                white-space:nowrap; border:1px solid rgba(0,0,0,0.15);
                pointer-events:none;">
                {area_name}
            </div>""",
            icon_size=(200, 30),
            icon_anchor=(100, 15),
        ),
    ).add_to(m)

# ── 6. Legenda manual ────────────────────────────────────────────────────────
legend_html = """
<div style="position:absolute; bottom:30px; left:20px; z-index:9999;
     background:white; padding:10px 14px; border-radius:8px;
     border:1px solid rgba(0,0,0,0.15); font-family:sans-serif; font-size:12px;">
  <b style="font-size:13px;">Áreas — Força Municipal</b><br><br>
"""
for area, color in color_map.items():
    legend_html += f"""
  <div style="display:flex; align-items:center; gap:7px; margin-bottom:5px;">
    <div style="width:14px;height:14px;border-radius:3px;background:{color};flex-shrink:0;"></div>
    <span>{area}</span>
  </div>"""
legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

# ── 7. Exibe no notebook ──────────────────────────────────────────────────────
display(m)

# ── 8. Opcional: salva como HTML standalone ───────────────────────────────────
m.save("mapa_areas_forca_municipal.html")
print("Mapa salvo em mapa_areas_forca_municipal.html")

Coluna de nome: 'nome_subar'
Áreas encontradas: ['Rodoviária - Terminal Gentileza - Estação Leopoldina', 'Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria', 'Jardim de Alah', 'Campo Grande: Estação de Trem - Calçadão', 'Rio Sul', 'Praia de Botafogo - Rua Marquês de Abrantes', 'Estações São Francisco Xavier - Afonso Pena', 'Presidente Vargas - Campo de Santana - Central do Brasil - Cinelândia']


Mapa salvo em mapa_areas_forca_municipal.html


Criando score

In [18]:
import pandas as pd
import numpy as np

# ── 1. Colunas a dropar ───────────────────────────────────────────────────────
DROP_COLS = [
    "estado", "municipio", "cep_logradouro",
    "referencia_logradouro", "complemento_logradouro", "tipo_logradouro",
]
df = df_denuncias.drop(columns=[c for c in DROP_COLS if c in df_denuncias.columns]).copy()

# ── 2. Filtra apenas ocorrências com polígono e coordenadas ──────────────────
df = df[
    df["area_forca_municipal"].notna() &
    ~df["area_forca_municipal"].isin(["Sem coordenadas", "Fora da área de operação"])
].copy()

print(f"Ocorrências com polígono: {len(df):,}")

# ── 3. Feature de turno (6 momentos do dia) ───────────────────────────────────
TURNOS = {
    "manha_1"  : (5,  9),    # 05h–08h59  — início do dia
    "manha_2"  : (9,  12),   # 09h–11h59  — manhã plena
    "tarde_1"  : (12, 15),   # 12h–14h59  — início da tarde
    "tarde_2"  : (15, 18),   # 15h–17h59  — tarde
    "noite_1"  : (18, 22),   # 18h–21h59  — início da noite
    "noite_2"  : (22, 5),    # 22h–04h59  — madrugada (wrap-around)
}

def atribuir_turno(hora: int) -> str:
    for turno, (ini, fim) in TURNOS.items():
        if ini < fim:
            if ini <= hora < fim:
                return turno
        else:                          # wrap-around (noite_2)
            if hora >= ini or hora < fim:
                return turno
    return "noite_2"

df["hora"]  = df["data_denuncia"].dt.hour
df["turno"] = df["hora"].apply(atribuir_turno)

# ── 4. Colunas temporais ──────────────────────────────────────────────────────
df["date"]        = df["data_denuncia"].dt.normalize()          # dia (sem hora)
df["ano_semana"]  = df["data_denuncia"].dt.isocalendar().year.astype(int)
df["semana"]      = df["data_denuncia"].dt.isocalendar().week.astype(int)
df["semana_inicio"] = df["data_denuncia"].dt.to_period("W").apply(lambda p: p.start_time)

# ── 5. Peso de cada ocorrência ────────────────────────────────────────────────
CRIME_WEIGHTS = {
    "POSSE  ILÍCITA DE  ARMAS FOGO"       : 9,
    "MAUS TRATOS"                          : 9,
    "ROUBO/FURTO A TRANSEUNTES"            : 8,
    "CORRUPÇÃO DE MENORES"                 : 8,
    "ROUBO A MOTORISTAS"                   : 8,
    "TRÁFICO DE DROGAS"                    : 7,
    "AMEAÇA"                               : 7,
    "CRIANÇA E ADOLESCENTE INFRATOR"       : 6,
    "RECEP/COMERC PROD ROUBADOS/FURTADOS"  : 6,
    "SUSPEITA DE ROUBO/FURTO"              : 5,
    "CONSUMO DE DROGAS"                    : 4,
    "BADERNA"                              : 3,
    "OBSTRUÇÃO DE VIAS PÚBLICAS"           : 3,
    "EST COMERCIAL/INDUSTRIAL SEM ALVARÁ"  : 3,
    "BARULHO"                              : 2,
}
DEFAULT_WEIGHT = 5

df["peso"] = df["tipo"].map(CRIME_WEIGHTS).fillna(DEFAULT_WEIGHT).astype(int)

print("\nAmostra com turno e peso:")
print(df[["data_denuncia","hora","turno","tipo","peso","area_forca_municipal"]].head(8).to_string())

Ocorrências com polígono: 772

Amostra com turno e peso:
          data_denuncia  hora    turno               tipo  peso                                           area_forca_municipal
3   2022-11-07 17:09:00    17  tarde_2  CONSUMO DE DROGAS     4                    Estações São Francisco Xavier - Afonso Pena
126 2023-08-14 10:20:00    10  manha_2  CONSUMO DE DROGAS     4                       Campo Grande: Estação de Trem - Calçadão
235 2021-08-17 20:53:00    20  noite_1  CONSUMO DE DROGAS     4  Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria
255 2019-05-21 07:36:00     7  manha_1  CONSUMO DE DROGAS     4  Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria
265 2021-11-29 13:04:00    13  tarde_1  CONSUMO DE DROGAS     4  Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria
467 2025-12-18 19:23:00    19  noite_1  CONSUMO DE DROGAS     4  Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria
505 2023-01-13 17:28:00    17  tarde_2  CONSUMO DE DRO

In [19]:
# ── 6. Agregação semanal por área ─────────────────────────────────────────────
# Score bruto da semana = soma dos pesos (captura volume + gravidade juntos)
weekly_raw = (
    df.groupby(["semana_inicio", "ano_semana", "semana", "area_forca_municipal"])
    .agg(
        ocorrencias        = ("peso", "count"),
        score_bruto        = ("peso", "sum"),
        peso_medio         = ("peso", "mean"),
        # distribuição por turno
        manha_1            = ("turno", lambda x: (x == "manha_1").sum()),
        manha_2            = ("turno", lambda x: (x == "manha_2").sum()),
        tarde_1            = ("turno", lambda x: (x == "tarde_1").sum()),
        tarde_2            = ("turno", lambda x: (x == "tarde_2").sum()),
        noite_1            = ("turno", lambda x: (x == "noite_1").sum()),
        noite_2            = ("turno", lambda x: (x == "noite_2").sum()),
        # distribuição por classe
        crimes_patrimonio  = ("classe", lambda x: (x.str.contains("PATRIMÔNIO", na=False)).sum()),
        crimes_pessoa      = ("classe", lambda x: (x.str.contains("PESSOA",    na=False)).sum()),
        crianca_adolesc    = ("classe", lambda x: (x.str.contains("CRIANÇA",   na=False)).sum()),
        substancias        = ("classe", lambda x: (x.str.contains("ENTORPEC",  na=False)).sum()),
        ordem_publica      = ("classe", lambda x: (x.str.contains("ORDEM",     na=False)).sum()),
        armas              = ("classe", lambda x: (x.str.contains("ARMAS",     na=False)).sum()),
    )
    .reset_index()
    .sort_values(["area_forca_municipal", "semana_inicio"])
    .reset_index(drop=True)
)

print(f"Weekly raw shape: {weekly_raw.shape}")
weekly_raw.head(6)

Weekly raw shape: (630, 19)


,semana_inicio,ano_semana,semana,area_forca_municipal,ocorrencias,score_bruto,peso_medio,manha_1,manha_2,tarde_1,tarde_2,noite_1,noite_2,crimes_patrimonio,crimes_pessoa,crianca_adolesc,substancias,ordem_publica,armas
0,2019-05-06,2019,19,Campo Grande: Estação de Trem - Calçadão,1,8,8.0,0,0,1,0,0,0,1,0,0,0,0,0
1,2019-06-03,2019,23,Campo Grande: Estação de Trem - Calçadão,2,12,6.0,0,0,0,0,2,0,1,0,0,1,0,0
2,2019-06-24,2019,26,Campo Grande: Estação de Trem - Calçadão,1,4,4.0,0,1,0,0,0,0,0,0,0,1,0,0
3,2019-07-01,2019,27,Campo Grande: Estação de Trem - Calçadão,1,8,8.0,0,1,0,0,0,0,1,0,0,0,0,0
4,2019-10-28,2019,44,Campo Grande: Estação de Trem - Calçadão,1,8,8.0,1,0,0,0,0,0,1,0,0,0,0,0
5,2020-11-16,2020,47,Campo Grande: Estação de Trem - Calçadão,1,4,4.0,0,0,0,0,1,0,0,0,0,1,0,0


In [20]:
# ── 7. Score com média móvel ponderada (últimas 8 semanas) ───────────────────
# Pesos exponenciais: semanas mais recentes têm maior influência
N_WEEKS  = 8
exp_w    = np.exp(np.linspace(0, 1, N_WEEKS))          # [e^0 .. e^1]
exp_w    /= exp_w.sum()                                 # normaliza → soma = 1

def rolling_ewma_score(series: pd.Series) -> pd.Series:
    """Média móvel exponencial manual com janela de N_WEEKS."""
    result = np.full(len(series), np.nan)
    vals   = series.values
    for i in range(len(vals)):
        if i < N_WEEKS - 1:
            # janela incompleta: usa os dados disponíveis com pesos re-normalizados
            window  = vals[:i+1]
            w_slice = exp_w[-len(window):]
            w_slice = w_slice / w_slice.sum()
            result[i] = np.dot(window, w_slice)
        else:
            window   = vals[i - N_WEEKS + 1 : i + 1]
            result[i] = np.dot(window, exp_w)
    return pd.Series(result, index=series.index)

weekly_raw["score"] = (
    weekly_raw
    .groupby("area_forca_municipal")["score_bruto"]
    .transform(rolling_ewma_score)
    .round(2)
)

print("Score calculado. Amostra:")
print(weekly_raw[["semana_inicio","area_forca_municipal","ocorrencias","score_bruto","score"]].head(12).to_string())

Score calculado. Amostra:
   semana_inicio                      area_forca_municipal  ocorrencias  score_bruto  score
0     2019-05-06  Campo Grande: Estação de Trem - Calçadão            1            8   8.00
1     2019-06-03  Campo Grande: Estação de Trem - Calçadão            2           12  10.14
2     2019-06-24  Campo Grande: Estação de Trem - Calçadão            1            4   7.80
3     2019-07-01  Campo Grande: Estação de Trem - Calçadão            1            8   7.86
4     2019-10-28  Campo Grande: Estação de Trem - Calçadão            1            8   7.90
5     2020-11-16  Campo Grande: Estação de Trem - Calçadão            1            4   6.99
6     2021-01-04  Campo Grande: Estação de Trem - Calçadão            1            4   6.36
7     2021-01-11  Campo Grande: Estação de Trem - Calçadão            1            4   5.90
8     2021-05-17  Campo Grande: Estação de Trem - Calçadão            1            4   5.40
9     2021-06-21  Campo Grande: Estação de Trem - Calç

In [21]:
# ── 8. Série temporal final ───────────────────────────────────────────────────
FINAL_COLS = [
    "semana_inicio",          # datetime da segunda-feira da semana
    "ano_semana",
    "semana",
    "area_forca_municipal",   # região
    "ocorrencias",            # volume bruto
    "score_bruto",            # soma pesos na semana
    "score",                  # média móvel exp. 8 semanas  ← valor principal
    "peso_medio",
    "manha_1","manha_2",
    "tarde_1","tarde_2",
    "noite_1","noite_2",
    "crimes_patrimonio","crimes_pessoa","crianca_adolesc",
    "substancias","ordem_publica","armas",
]

df_weekly = weekly_raw[FINAL_COLS].copy()
df_weekly  = df_weekly.rename(columns={"semana_inicio": "datetime"})

print(f"\ndf_weekly shape : {df_weekly.shape}")
print(f"Período         : {df_weekly['datetime'].min().date()} → {df_weekly['datetime'].max().date()}")
print(f"Áreas           : {df_weekly['area_forca_municipal'].nunique()}")
print(f"Semanas         : {df_weekly['datetime'].nunique()}")
print()
df_weekly.info()
print()
df_weekly.head(10)


df_weekly shape : (630, 20)
Período         : 2018-12-31 → 2026-05-04
Áreas           : 8
Semanas         : 316

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630 entries, 0 to 629
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              630 non-null    datetime64[ns]
 1   ano_semana            630 non-null    int32         
 2   semana                630 non-null    int32         
 3   area_forca_municipal  630 non-null    object        
 4   ocorrencias           630 non-null    int64         
 5   score_bruto           630 non-null    int32         
 6   score                 630 non-null    float64       
 7   peso_medio            630 non-null    float64       
 8   manha_1               630 non-null    int64         
 9   manha_2               630 non-null    int64         
 10  tarde_1               630 non-null    int64         
 11  tarde_2               

,datetime,ano_semana,semana,area_forca_municipal,ocorrencias,score_bruto,score,peso_medio,manha_1,manha_2,tarde_1,tarde_2,noite_1,noite_2,crimes_patrimonio,crimes_pessoa,crianca_adolesc,substancias,ordem_publica,armas
0,2019-05-06,2019,19,Campo Grande: Estação de Trem - Calçadão,1,8,8.00,8.0,0,0,1,0,0,0,1,0,0,0,0,0
1,2019-06-03,2019,23,Campo Grande: Estação de Trem - Calçadão,2,12,10.14,6.0,0,0,0,0,2,0,1,0,0,1,0,0
2,2019-06-24,2019,26,Campo Grande: Estação de Trem - Calçadão,1,4,7.80,4.0,0,1,0,0,0,0,0,0,0,1,0,0
3,2019-07-01,2019,27,Campo Grande: Estação de Trem - Calçadão,1,8,7.86,8.0,0,1,0,0,0,0,1,0,0,0,0,0
4,2019-10-28,2019,44,Campo Grande: Estação de Trem - Calçadão,1,8,7.90,8.0,1,0,0,0,0,0,1,0,0,0,0,0
5,2020-11-16,2020,47,Campo Grande: Estação de Trem - Calçadão,1,4,6.99,4.0,0,0,0,0,1,0,0,0,0,1,0,0
6,2021-01-04,2021,1,Campo Grande: Estação de Trem - Calçadão,1,4,6.36,4.0,0,0,0,1,0,0,0,0,0,1,0,0
7,2021-01-11,2021,2,Campo Grande: Estação de Trem - Calçadão,1,4,5.90,4.0,0,0,0,1,0,0,0,0,0,1,0,0
8,2021-05-17,2021,20,Campo Grande: Estação de Trem - Calçadão,1,4,5.40,4.0,0,0,0,0,1,0,0,0,0,1,0,0
9,2021-06-21,2021,25,Campo Grande: Estação de Trem - Calçadão,1,8,5.50,8.0,0,1,0,0,0,0,1,0,0,0,0,0


In [22]:
# ── 8. Série temporal final ───────────────────────────────────────────────────
FINAL_COLS = [
    "semana_inicio",          # datetime da segunda-feira da semana
    "ano_semana",
    "semana",
    "area_forca_municipal",   # região
    "ocorrencias",            # volume bruto
    "score_bruto",            # soma pesos na semana
    "score",                  # média móvel exp. 8 semanas  ← valor principal
    "peso_medio",
    "manha_1","manha_2",
    "tarde_1","tarde_2",
    "noite_1","noite_2",
    "crimes_patrimonio","crimes_pessoa","crianca_adolesc",
    "substancias","ordem_publica","armas",
]

df_weekly = weekly_raw[FINAL_COLS].copy()
df_weekly  = df_weekly.rename(columns={"semana_inicio": "datetime"})

print(f"\ndf_weekly shape : {df_weekly.shape}")
print(f"Período         : {df_weekly['datetime'].min().date()} → {df_weekly['datetime'].max().date()}")
print(f"Áreas           : {df_weekly['area_forca_municipal'].nunique()}")
print(f"Semanas         : {df_weekly['datetime'].nunique()}")
print()
df_weekly.info()
print()
df_weekly.head(10)


df_weekly shape : (630, 20)
Período         : 2018-12-31 → 2026-05-04
Áreas           : 8
Semanas         : 316

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630 entries, 0 to 629
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              630 non-null    datetime64[ns]
 1   ano_semana            630 non-null    int32         
 2   semana                630 non-null    int32         
 3   area_forca_municipal  630 non-null    object        
 4   ocorrencias           630 non-null    int64         
 5   score_bruto           630 non-null    int32         
 6   score                 630 non-null    float64       
 7   peso_medio            630 non-null    float64       
 8   manha_1               630 non-null    int64         
 9   manha_2               630 non-null    int64         
 10  tarde_1               630 non-null    int64         
 11  tarde_2               

,datetime,ano_semana,semana,area_forca_municipal,ocorrencias,score_bruto,score,peso_medio,manha_1,manha_2,tarde_1,tarde_2,noite_1,noite_2,crimes_patrimonio,crimes_pessoa,crianca_adolesc,substancias,ordem_publica,armas
0,2019-05-06,2019,19,Campo Grande: Estação de Trem - Calçadão,1,8,8.00,8.0,0,0,1,0,0,0,1,0,0,0,0,0
1,2019-06-03,2019,23,Campo Grande: Estação de Trem - Calçadão,2,12,10.14,6.0,0,0,0,0,2,0,1,0,0,1,0,0
2,2019-06-24,2019,26,Campo Grande: Estação de Trem - Calçadão,1,4,7.80,4.0,0,1,0,0,0,0,0,0,0,1,0,0
3,2019-07-01,2019,27,Campo Grande: Estação de Trem - Calçadão,1,8,7.86,8.0,0,1,0,0,0,0,1,0,0,0,0,0
4,2019-10-28,2019,44,Campo Grande: Estação de Trem - Calçadão,1,8,7.90,8.0,1,0,0,0,0,0,1,0,0,0,0,0
5,2020-11-16,2020,47,Campo Grande: Estação de Trem - Calçadão,1,4,6.99,4.0,0,0,0,0,1,0,0,0,0,1,0,0
6,2021-01-04,2021,1,Campo Grande: Estação de Trem - Calçadão,1,4,6.36,4.0,0,0,0,1,0,0,0,0,0,1,0,0
7,2021-01-11,2021,2,Campo Grande: Estação de Trem - Calçadão,1,4,5.90,4.0,0,0,0,1,0,0,0,0,0,1,0,0
8,2021-05-17,2021,20,Campo Grande: Estação de Trem - Calçadão,1,4,5.40,4.0,0,0,0,0,1,0,0,0,0,1,0,0
9,2021-06-21,2021,25,Campo Grande: Estação de Trem - Calçadão,1,8,5.50,8.0,0,1,0,0,0,0,1,0,0,0,0,0


## Relato da ocorrencia:


In [23]:
# ── Agrupa relatos por semana + área ─────────────────────────────────────────
def formatar_relatos(grupo: pd.DataFrame) -> str:
    """
    Formata todos os relatos do grupo como uma lista numerada estruturada,
    prefixada com metadados úteis para um agente de IA resumir.
    """
    linhas = []
    for i, (_, row) in enumerate(grupo.iterrows(), 1):
        turno  = row.get("turno", "?")
        tipo   = row.get("tipo",  "?")
        classe = row.get("classe","?")
        peso   = row.get("peso",  "?")
        bairro = row.get("bairro_logradouro", "?")
        relato = str(row.get("relato_redacted", "")).strip()
        if not relato or relato.lower() in ("nan", "none", ""):
            relato = "SEM RELATO"
        linhas.append(
            f"[{i}] TURNO:{turno} | BAIRRO:{bairro} | CLASSE:{classe} "
            f"| TIPO:{tipo} | PESO:{peso}\n    RELATO: {relato}"
        )
    return "\n".join(linhas)


# ── Agrega ────────────────────────────────────────────────────────────────────
df_relatos = (
    df                                          # df já tem turno, peso, area
    .groupby(
        ["semana_inicio", "ano_semana", "semana", "area_forca_municipal"],
        sort=True,
    )
    .apply(
        lambda g: pd.Series({
            "datetime"            : g["semana_inicio"].iloc[0],
            "ocorrencias"         : len(g),
            "score_bruto"         : int(g["peso"].sum()),
            "tipos_unicos"        : ", ".join(sorted(g["tipo"].dropna().unique())),
            "bairros_afetados"    : ", ".join(sorted(g["bairro_logradouro"].dropna().unique())),
            "relatos_estruturados": formatar_relatos(g),
        })
    )
    .reset_index()
    .drop(columns=["semana_inicio"])            # já está em datetime
    .sort_values(["area_forca_municipal", "datetime"])
    .reset_index(drop=True)
)

# ── Reordena colunas ──────────────────────────────────────────────────────────
df_relatos = df_relatos[[
    "datetime",
    "ano_semana",
    "semana",
    "area_forca_municipal",
    "ocorrencias",
    "score_bruto",
    "tipos_unicos",
    "bairros_afetados",
    "relatos_estruturados",
]]

print(f"df_relatos shape : {df_relatos.shape}")
print(f"Período          : {df_relatos['datetime'].min().date()} → {df_relatos['datetime'].max().date()}")
print(f"Áreas            : {df_relatos['area_forca_municipal'].nunique()}")
print()
df_relatos.info()

df_relatos shape : (630, 9)
Período          : 2018-12-31 → 2026-05-04
Áreas            : 8

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630 entries, 0 to 629
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              630 non-null    datetime64[ns]
 1   ano_semana            630 non-null    int32         
 2   semana                630 non-null    int32         
 3   area_forca_municipal  630 non-null    object        
 4   ocorrencias           630 non-null    int64         
 5   score_bruto           630 non-null    int64         
 6   tipos_unicos          630 non-null    object        
 7   bairros_afetados      630 non-null    object        
 8   relatos_estruturados  630 non-null    object        
dtypes: datetime64[ns](1), int32(2), int64(2), object(4)
memory usage: 39.5+ KB


In [24]:
# ── Merge com df_weekly para ter o score de média móvel junto ─────────────────
df_final = df_weekly[[
    "datetime", "ano_semana", "semana",
    "area_forca_municipal", "score",
    "manha_1","manha_2","tarde_1","tarde_2","noite_1","noite_2",
]].merge(
    df_relatos[[
        "datetime","ano_semana","semana","area_forca_municipal",
        "ocorrencias","score_bruto","tipos_unicos",
        "bairros_afetados","relatos_estruturados",
    ]],
    on=["datetime","ano_semana","semana","area_forca_municipal"],
    how="inner",
)

print(f"df_final shape: {df_final.shape}")
df_final.head(3)

df_final shape: (630, 16)


,datetime,ano_semana,semana,area_forca_municipal,score,manha_1,manha_2,tarde_1,tarde_2,noite_1,noite_2,ocorrencias,score_bruto,tipos_unicos,bairros_afetados,relatos_estruturados
0,2019-05-06,2019,19,Campo Grande: Estação de Trem - Calçadão,8.00,0,0,1,0,0,0,1,8,ROUBO/FURTO A TRANSEUNTES,CAMPO GRANDE,[1] TURNO:tarde_1 | BAIRRO:CAMPO GRANDE | CLAS...
1,2019-06-03,2019,23,Campo Grande: Estação de Trem - Calçadão,10.14,0,0,0,0,2,0,2,12,"CONSUMO DE DROGAS, ROUBO/FURTO A TRANSEUNTES",CAMPO GRANDE,[1] TURNO:noite_1 | BAIRRO:CAMPO GRANDE | CLAS...
2,2019-06-24,2019,26,Campo Grande: Estação de Trem - Calçadão,7.80,0,1,0,0,0,0,1,4,CONSUMO DE DROGAS,CAMPO GRANDE,[1] TURNO:manha_2 | BAIRRO:CAMPO GRANDE | CLAS...


In [25]:
# ── Exporta ───────────────────────────────────────────────────────────────────

# Parquet — preserva tipos e é eficiente para colunas de texto longas
df_final.to_parquet("denuncias_semanal_com_relatos.parquet", index=False)

# CSV com separador pipe para não conflitar com vírgulas nos relatos
df_final.to_csv("denuncias_semanal_com_relatos.csv", index=False, sep="|")

print("Exportado:")
print("  denuncias_semanal_com_relatos.parquet")
print("  denuncias_semanal_com_relatos.csv")
print()
print("Colunas finais:")
for c in df_final.columns:
    print(f"  {c:<30} {str(df_final[c].dtype):<20} ex: {str(df_final[c].iloc[0])[:60]}")

Exportado:
  denuncias_semanal_com_relatos.parquet
  denuncias_semanal_com_relatos.csv

Colunas finais:
  datetime                       datetime64[ns]       ex: 2019-05-06 00:00:00
  ano_semana                     int32                ex: 2019
  semana                         int32                ex: 19
  area_forca_municipal           object               ex: Campo Grande: Estação de Trem - Calçadão
  score                          float64              ex: 8.0
  manha_1                        int64                ex: 0
  manha_2                        int64                ex: 0
  tarde_1                        int64                ex: 1
  tarde_2                        int64                ex: 0
  noite_1                        int64                ex: 0
  noite_2                        int64                ex: 0
  ocorrencias                    int64                ex: 1
  score_bruto                    int64                ex: 8
  tipos_unicos                   object              